# MODULE 3 : Développement Applicatif
## VII : Déploiement de l'application

**Rappel du contexte :** Votre application est développée, testée, fiabilisée. Reste une question très concrète : **comment le service commercial va-t-il l'utiliser ?** Il n'a pas Python installé, ne sait pas ce qu'est un terminal, et ne doit surtout pas avoir à taper `python interface_gestion.py`. Aujourd'hui : le **déploiement**.


### Objectifs du jour
- Comprendre les enjeux du déploiement (dépendances, environnements cibles)
- Empaqueter l'application en exécutable autonome avec PyInstaller
- Externaliser la configuration sensible (mot de passe BD) hors du code source
- Installer et tester l'application sur un poste distant, connecté à une base de données réseau
- Rédiger une procédure d'installation utilisable par un non-développeur

### Déroulé de la journée
1. Les enjeux du déploiement
2. Sécuriser la configuration (variables d'environnement)
3. Packaging avec PyInstaller
4. Déploiement réseau : base MySQL accessible depuis un autre poste
5. Atelier : procédure d'installation documentée


---
## 1. Les enjeux du déploiement

### a) Le problème concret

Sur votre poste de développement, l'application fonctionne car :
- Python est installé
- Les bibliothèques (`mysql-connector-python`) sont installées
- Le fichier `gestion_commerciale.py` est dans le même dossier que `interface_gestion.py`
- Vous savez ouvrir un terminal et taper une commande

**Rien de tout ça n'est vrai sur le poste du service commercial.** Le déploiement consiste à combler cet écart.

### b) Les stratégies de déploiement courantes en entreprise

| Stratégie | Quand l'utiliser| 
|---|---|
| Exécutable autonome (PyInstaller, cx_Freeze) | Appli de bureau, poste utilisateur isolé |
| Conteneur (Docker) | Applications web, environnements serveurs reproductibles |
| Installateur (MSI, .deb) | Distribution à grande échelle, mises à jour gérées |
| Application web hébergée | Accessible depuis n'importe quel navigateur |

Pour une petite application de gestion interne comme la nôtre, **l'exécutable autonome + une base de données accessible en réseau** est une solution réaliste et couramment utilisée par les PME.

---
## 2. Sécuriser la configuration


### a)  Variables d'environnement

In [1]:
%%writefile config.py
"""
Configuration de l'application, lue depuis les variables d'environnement.
Ce fichier peut être partagé/versionné sans risque : il ne contient AUCUN secret.
"""
import os

def config_bd_depuis_environnement():
    """
    Construit la configuration de connexion à la base de données à partir
    des variables d'environnement, avec des valeurs par défaut raisonnables
    pour le développement local uniquement.
    """
    return {
        "host": os.environ.get("GESTION_BD_HOTE", "localhost"),
        "user": os.environ.get("GESTION_BD_UTILISATEUR", "root"),
        "password": os.environ.get("GESTION_BD_MOT_DE_PASSE", ""),
        "database": os.environ.get("GESTION_BD_NOM", "gestion_commerciale"),
    }

def verifier_configuration():
    """Vérifie que le mot de passe a bien été fourni (pas la valeur par défaut vide)."""
    config = config_bd_depuis_environnement()
    if not config["password"]:
        raise EnvironmentError(
            "La variable d'environnement GESTION_BD_MOT_DE_PASSE n'est pas définie.\n"
            "Définissez-la avant de lancer l'application (voir README d'installation)."
        )
    return config

Writing config.py


In [2]:
import subprocess

resultat = subprocess.run(
    ["python3", "-c", "from config import verifier_configuration; verifier_configuration()"],
    capture_output=True, text=True
)
print("STDERR :", resultat.stderr[-300:])
print("\n Avec la variable d'environnement définie \n")

import os
env = os.environ.copy()
env["GESTION_BD_MOT_DE_PASSE"] = "mon_mot_de_passe_secret"
resultat2 = subprocess.run(
    ["python3", "-c", "from config import verifier_configuration; print(verifier_configuration())"],
    capture_output=True, text=True, env=env
)
print("STDOUT :", resultat2.stdout)

STDERR : Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.


 Avec la variable d'environnement définie 

STDOUT : 


**Comment définir une variable d'environnement en pratique :**

*Windows (PowerShell) :*
```powershell
$env:GESTION_BD_MOT_DE_PASSE = "mon_mot_de_passe"
```

*Linux / macOS (terminal) :*
```bash
export GESTION_BD_MOT_DE_PASSE="mon_mot_de_passe"
```

*Pour une utilisation permanente (fichier `.env` + bibliothèque `python-dotenv`)* : solution plus confortable pour l'utilisateur final, à explorer si le temps le permet en autonomie.

---
## 3. Packaging avec PyInstaller

PyInstaller analyse votre script Python, détecte les bibliothèques utilisées, et produit un **exécutable autonome** qui embarque tout (y compris un interpréteur Python), l'utilisateur final n'a besoin d'installer ni Python ni aucune bibliothèque.

### a) Installation de l'outil
```bash
pip install pyinstaller
```

### b) Préparons l'application finale à empaqueter

On reprend `interface_gestion.py`, en le connectant à la configuration sécurisée.

In [ ]:
%%writefile interface_gestion.py
"""
Interface graphique Tkinter — version prête pour le déploiement.
Lancer avec : python interface_gestion.py
Nécessite la variable d'environnement GESTION_BD_MOT_DE_PASSE définie.
"""
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from gestion_commerciale import GestionCommerciale

class ApplicationGestion:
    def __init__(self, fenetre):
        self.gestion = GestionCommerciale()
        self.fenetre = fenetre
        self.fenetre.title("Gestion Clients / Commandes")
        self.fenetre.geometry("700x550")

        self._construire_formulaire()
        self._construire_recherche()
        self._construire_tableau()
        self._construire_actions()

    def _construire_formulaire(self):
        cadre = tk.LabelFrame(self.fenetre, text="Ajouter un client", padx=10, pady=10)
        cadre.pack(padx=10, pady=5, fill="x")
        tk.Label(cadre, text="Nom :").grid(row=0, column=0, sticky="e", padx=5, pady=3)
        self.champ_nom = tk.Entry(cadre, width=25)
        self.champ_nom.grid(row=0, column=1, padx=5, pady=3)
        tk.Label(cadre, text="Email :").grid(row=1, column=0, sticky="e", padx=5, pady=3)
        self.champ_email = tk.Entry(cadre, width=25)
        self.champ_email.grid(row=1, column=1, padx=5, pady=3)
        tk.Label(cadre, text="Téléphone :").grid(row=2, column=0, sticky="e", padx=5, pady=3)
        self.champ_telephone = tk.Entry(cadre, width=25)
        self.champ_telephone.grid(row=2, column=1, padx=5, pady=3)
        tk.Button(cadre, text="Ajouter", command=self.ajouter_client).grid(
            row=3, column=0, columnspan=2, pady=8)

    def _construire_recherche(self):
        cadre = tk.Frame(self.fenetre, padx=10)
        cadre.pack(fill="x")
        tk.Label(cadre, text="Rechercher :").pack(side="left")
        self.champ_recherche = tk.Entry(cadre, width=30)
        self.champ_recherche.pack(side="left", padx=5)
        self.champ_recherche.bind("<KeyRelease>", lambda event: self._rafraichir_tableau())

    def _construire_tableau(self):
        cadre = tk.LabelFrame(self.fenetre, text="Liste des clients", padx=10, pady=10)
        cadre.pack(padx=10, pady=5, fill="both", expand=True)
        colonnes = ("id", "nom", "email", "telephone", "total_depense")
        self.tableau = ttk.Treeview(cadre, columns=colonnes, show="headings")
        libelles = {"id": "ID", "nom": "Nom", "email": "Email",
                    "telephone": "Téléphone", "total_depense": "Total dépensé (€)"}
        for col in colonnes:
            self.tableau.heading(col, text=libelles[col])
            self.tableau.column(col, width=120)
        self.tableau.pack(fill="both", expand=True)

    def _construire_actions(self):
        cadre = tk.Frame(self.fenetre, pady=10)
        cadre.pack(fill="x")
        tk.Button(cadre, text="Exporter en CSV", command=self.exporter_csv).pack(side="left", padx=10)
        tk.Button(cadre, text="Supprimer le client sélectionné", command=self.supprimer_client).pack(
            side="left", padx=10)

    def ajouter_client(self):
        nom = self.champ_nom.get()
        email = self.champ_email.get()
        telephone = self.champ_telephone.get()
        try:
            self.gestion.ajouter_client(nom, email, telephone)
        except ValueError as erreur:
            messagebox.showerror("Erreur de saisie", str(erreur))
            return
        self._rafraichir_tableau()
        self.champ_nom.delete(0, tk.END)
        self.champ_email.delete(0, tk.END)
        self.champ_telephone.delete(0, tk.END)
        messagebox.showinfo("Succès", "Client ajouté avec succès.")

    def supprimer_client(self):
        selection = self.tableau.selection()
        if not selection:
            messagebox.showwarning("Attention", "Sélectionnez un client dans le tableau.")
            return
        id_client = int(self.tableau.item(selection[0])["values"][0])
        try:
            self.gestion.supprimer_client(id_client)
        except ValueError as erreur:
            messagebox.showerror("Suppression impossible", str(erreur))
            return
        self._rafraichir_tableau()

    def exporter_csv(self):
        chemin = filedialog.asksaveasfilename(defaultextension=".csv",
                                               filetypes=[("Fichier CSV", "*.csv")])
        if not chemin:
            return
        self.gestion.exporter_clients_csv(chemin)
        messagebox.showinfo("Export réussi", f"Clients exportés vers :\n{chemin}")

    def _rafraichir_tableau(self):
        for ligne in self.tableau.get_children():
            self.tableau.delete(ligne)
        terme = self.champ_recherche.get()
        for client in self.gestion.rechercher_clients(terme):
            self.tableau.insert("", tk.END, values=(
                client.id_client, client.nom, client.email,
                client.telephone, client.total_depense()
            ))


def demarrer_application():
    """Point d'entrée principal, utilisé aussi bien en dev qu'en exécutable packagé."""
    fenetre = tk.Tk()
    app = ApplicationGestion(fenetre)
    fenetre.mainloop()


if __name__ == "__main__":
    demarrer_application()

### c) Générer l'exécutable


```bash
pyinstaller --onefile --windowed --name GestionCommerciale interface_gestion.py
```

**Décryptage des options :**

| Option | Rôle |
|---|---|
| `--onefile` | Produit un **seul** fichier exécutable (plus simple à distribuer qu'un dossier) |
| `--windowed` | N'ouvre PAS de console noire derrière la fenêtre Tkinter (indispensable pour une appli grand public) |
| `--name` | Nom du fichier exécutable produit |

**Résultat :** un dossier `dist/` apparaît, contenant :
- `GestionCommerciale.exe` (Windows) ou `GestionCommerciale` (Linux/Mac)

C'est **ce seul fichier** qu'on donnera au service commercial, pas besoin qu'il installe Python, ni qu'il voie le code source.

### d) Vérification du mécanisme (sur un script sans interface graphique)

Tkinter ne peut pas s'exécuter dans cet environnement de démonstration (pas d'affichage graphique), mais on peut vérifier concrètement que PyInstaller fonctionne, sur un petit script de test équivalent en logique.

In [ ]:
%%writefile demo_verification_packaging.py
"""Petit script de démonstration pour vérifier concrètement le mécanisme PyInstaller."""
from gestion_commerciale import GestionCommerciale

gestion = GestionCommerciale()
client = gestion.ajouter_client("Démo Packaging", "demo@test.com", "0611223344")
print(f"Application empaquetée fonctionnelle : client créé -> {client}")
print("Si ce message s'affiche depuis l'exécutable dist/, le packaging a réussi.")

In [ ]:
import subprocess

# Génère l'exécutable (équivalent à la commande du terminal, exécutée ici pour vérification)
resultat = subprocess.run(
    ["pyinstaller", "--onefile", "--name", "GestionCommerciale_demo", "demo_verification_packaging.py"],
    capture_output=True, text=True
)
print("Code retour :", resultat.returncode)
print(resultat.stdout[-500:])

In [ ]:
import os

print("Contenu de dist/ :", os.listdir("dist"))

# Exécution de l'exécutable généré, EXACTEMENT comme le ferait un utilisateur final
resultat = subprocess.run(["./dist/GestionCommerciale_demo"], capture_output=True, text=True)
print("\n Sortie de l'exécutable packagé ")
print(resultat.stdout)

**Ce que vous venez de vérifier :** l'exécutable produit par PyInstaller fonctionne de façon totalement autonome — il embarque l'interpréteur Python et toutes les dépendances. Un poste sans Python installé peut l'exécuter directement.
```bash
pyinstaller --onefile --windowed --name GestionCommerciale interface_gestion.py
./dist/GestionCommerciale
```

In [ ]:
# Nettoyage des fichiers générés par la démonstration (bonne pratique : ne pas polluer le dépôt)
import shutil
for dossier in ["dist", "build"]:
    if os.path.exists(dossier):
        shutil.rmtree(dossier)
for fichier in ["GestionCommerciale_demo.spec", "demo_verification_packaging.py"]:
    if os.path.exists(fichier):
        os.remove(fichier)
print("Nettoyage effectué.")

---
## 4. Déploiement réseau : base MySQL accessible depuis un autre poste

L'exécutable est prêt, mais il doit se connecter à **une base de données accessible depuis le poste où il tourne**. Deux cas de figure réalistes en PME :

### a) Cas A : Base de données centralisée sur un serveur du réseau local

C'est le cas le plus courant : un serveur MySQL tourne sur une machine dédiée (ou le poste d'un administrateur), et tous les postes du service commercial s'y connectent.

**Étapes côté serveur (à faire une seule fois, par l'administrateur) :**

1. Autoriser les connexions distantes dans la configuration MySQL (`my.cnf` ou `my.ini`) :
```ini
bind-address = 0.0.0.0
```

2. Créer un utilisateur MySQL dédié à l'application, avec un mot de passe fort, autorisé à se connecter depuis le réseau local (et pas depuis n'importe où sur Internet) :
```sql
CREATE USER 'appli_gestion'@'192.168.1.%' IDENTIFIED BY 'mot_de_passe_fort';
GRANT SELECT, INSERT, UPDATE, DELETE ON gestion_commerciale.* TO 'appli_gestion'@'192.168.1.%';
FLUSH PRIVILEGES;
```

3. Vérifier que le pare-feu du serveur autorise le port MySQL (3306) depuis le réseau local uniquement.

**Étapes côté poste client (chaque poste du service commercial) :**

Définir les variables d'environnement pointant vers le serveur (pas `localhost` cette fois, mais l'adresse IP du serveur) :
```bash
GESTION_BD_HOTE=192.168.1.50
GESTION_BD_UTILISATEUR=appli_gestion
GESTION_BD_MOT_DE_PASSE=mot_de_passe_fort
GESTION_BD_NOM=gestion_commerciale
```

### b) Cas B : Test d'installation sur un second poste (exercice)

Si votre centre de formation dispose de plusieurs postes en réseau, réalisez ce test grandeur nature :

1. Sur le **poste A** (serveur) : la base MySQL tourne, avec les autorisations ci-dessus
2. Sur le **poste B** (client) : copiez uniquement `GestionCommerciale.exe` (l'exécutable, PAS le code source)
3. Définissez les variables d'environnement sur le poste B avec l'adresse IP du poste A
4. Lancez l'exécutable sur le poste B et vérifiez que les clients ajoutés apparaissent bien dans la base du poste A (demandez à un camarade de vérifier depuis un client MySQL)

---
## 5.  Procédure d'installation documentée

Un développeur professionnel ne livre jamais "juste un fichier", il livre **le fichier + une procédure claire**, utilisable par quelqu'un qui n'a jamais vu le projet.

In [3]:
%%writefile PROCEDURE_INSTALLATION.md
# Procédure d'installation — Application Gestion Clients/Commandes

## Prérequis
- Système Windows 10/11 (ou Linux/macOS selon la version fournie)
- Accès réseau au serveur de base de données (adresse fournie par l'administrateur)

## Étape 1 — Copier l'application
Copiez le fichier `GestionCommerciale.exe` dans un dossier de votre choix
(par exemple `C:\Applications\GestionCommerciale\`).

## Étape 2 — Configurer la connexion à la base de données
Créez un fichier `demarrer.bat` dans le même dossier, avec le contenu suivant
(remplacez les valeurs entre `<>` par celles fournies par votre administrateur) :

```bat
@echo off
set GESTION_BD_HOTE=<adresse IP du serveur>
set GESTION_BD_UTILISATEUR=<utilisateur fourni>
set GESTION_BD_MOT_DE_PASSE=<mot de passe fourni>
set GESTION_BD_NOM=gestion_commerciale
GestionCommerciale.exe
```

## Étape 3 — Lancer l'application
Double-cliquez sur `demarrer.bat`.

## En cas de problème
| Symptôme | Cause probable | Solution |
|---|---|---|
| L'application ne s'ouvre pas du tout | Antivirus bloquant l'exécutable | Ajouter une exception dans l'antivirus |
| Message "Impossible de se connecter à la base" | Mauvaise adresse IP ou mot de passe | Vérifier `demarrer.bat` avec l'administrateur |
| Message "variable d'environnement non définie" | `demarrer.bat` non utilisé (double-clic direct sur le .exe) | Toujours lancer via `demarrer.bat` |

## Contact support
En cas de problème persistant, contactez : <à compléter par le stagiaire>

Overwriting PROCEDURE_INSTALLATION.md


> **Ce document est important pour votre portfolio de stage :** savoir rédiger une procédure d'installation claire, anticipant les problèmes courants, est une compétence très valorisée, elle démontre que vous pensez à l'utilisateur final, pas seulement au code.